In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE TABLE FINANCE.CURATED.MERCHANT_RULES (
  priority INT,
  ilike_pattern STRING,          -- e.g. '%AMZN%' or '%COSTCO%'
  merchant_canonical STRING
);

TRUNCATE TABLE FINANCE.CURATED.MERCHANT_RULES;

INSERT INTO FINANCE.CURATED.MERCHANT_RULES (priority, ilike_pattern, merchant_canonical) VALUES
  (10, '%AMAZON%',      'AMAZON'),
  (11, '%AMZN%',        'AMAZON'),
  (20, '%COSTCO%',      'COSTCO'),
  (30, '%WHOLEFDS%',    'WHOLE FOODS'),
  (31, '%WHOLE FOODS%', 'WHOLE FOODS'),
  (40, '%APPLE%',       'APPLE'),
  (41, '%APPLE.COM/BILL%','APPLE'),
  (50, '%PRIME VIDEO%', 'PRIME VIDEO'),
  (60, '%UBER%',        'UBER'),
  (70, '%GRAB%',        'GRAB');

CREATE OR REPLACE TABLE FINANCE.CURATED.TXN_FEATURES AS
SELECT
  source, account_id, txn_date, post_date,
  description_raw, category_raw,
  amount_signed, is_payment,
  statement_merchant, country, chase_type,

  /* merchant_hint: normalized merchant-ish text */
  TRIM(
    REGEXP_REPLACE(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          UPPER(COALESCE(NULLIF(statement_merchant,''), description_raw)),
          '[0-9]', ' '
        ),
        '[^A-Z ]', ' '
      ),
      '\\s+', ' '
    )
  ) AS merchant_hint
FROM FINANCE.STG.TRANSACTIONS
WHERE post_date IS NOT NULL
  AND amount_signed IS NOT NULL;

CREATE OR REPLACE VIEW FINANCE.CURATED.MERCHANT_HINT_STATS AS
SELECT
  merchant_hint,
  COUNT(*) AS txn_count,
  SUM(amount_signed) AS spend,
  MIN(post_date) AS first_seen,
  MAX(post_date) AS last_seen
FROM FINANCE.CURATED.TXN_FEATURES
WHERE amount_signed > 0 AND NOT is_payment
  AND merchant_hint IS NOT NULL AND merchant_hint <> ''
GROUP BY 1
HAVING COUNT(*) >= 2
ORDER BY spend DESC;


In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Pull top hints by activity (cap to keep it fast)
df = session.sql("""
SELECT merchant_hint, txn_count, spend
FROM FINANCE.CURATED.MERCHANT_HINT_STATS
ORDER BY spend DESC
LIMIT 2000
""").to_pandas()

# Simple text similarity clustering via TF-IDF + cosine similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = df["MERCHANT_HINT"].fillna("").tolist()
vec = TfidfVectorizer(ngram_range=(1,2), min_df=2)
X = vec.fit_transform(texts)

sim = cosine_similarity(X)

# Greedy clustering: take each item, group highly similar ones not yet assigned
threshold = 0.72
cluster_id = [-1] * len(texts)
cid = 0
for i in range(len(texts)):
    if cluster_id[i] != -1:
        continue
    # find neighbors
    neighbors = [j for j in range(len(texts)) if sim[i, j] >= threshold]
    for j in neighbors:
        cluster_id[j] = cid
    cid += 1

df["CLUSTER_ID"] = cluster_id

# Pick a representative label per cluster (highest spend)
rep = (
    df.sort_values(["CLUSTER_ID","SPEND"], ascending=[True,False])
      .groupby("CLUSTER_ID")
      .head(1)[["CLUSTER_ID","MERCHANT_HINT","SPEND"]]
      .rename(columns={"MERCHANT_HINT":"REPRESENTATIVE_HINT","SPEND":"REP_SPEND"})
)

out = df.merge(rep, on="CLUSTER_ID", how="left")
out = out.sort_values(["REP_SPEND","CLUSTER_ID","SPEND"], ascending=[False,True,False])

# Write suggestions back to Snowflake
session.write_pandas(
    out[["CLUSTER_ID","REPRESENTATIVE_HINT","MERCHANT_HINT","TXN_COUNT","SPEND"]],
    "MERCHANT_CLUSTER_SUGGESTIONS",
    database="FINANCE",
    schema="CURATED",
    auto_create_table=True,
    overwrite=True
)


In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE TABLE FINANCE.CURATED.MERCHANT_CANONICAL_MAP (
  merchant_hint STRING PRIMARY KEY,
  merchant_canonical STRING,
  cluster_id NUMBER,
  updated_at TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP()
);

MERGE INTO FINANCE.CURATED.MERCHANT_CANONICAL_MAP m
USING (
  SELECT
    MERCHANT_HINT AS merchant_hint,
    REPRESENTATIVE_HINT AS merchant_canonical,
    CLUSTER_ID AS cluster_id
  FROM FINANCE.CURATED.MERCHANT_CLUSTER_SUGGESTIONS
) s
ON m.merchant_hint = s.merchant_hint
WHEN MATCHED THEN UPDATE SET
  merchant_canonical = s.merchant_canonical,
  cluster_id = s.cluster_id,
  updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (merchant_hint, merchant_canonical, cluster_id)
VALUES (s.merchant_hint, s.merchant_canonical, s.cluster_id);

CREATE OR REPLACE VIEW FINANCE.CURATED.TRANSACTIONS_ENRICHED AS
WITH base AS (
  SELECT
    t.*,
    TRIM(
      REGEXP_REPLACE(
        REGEXP_REPLACE(
          REGEXP_REPLACE(UPPER(COALESCE(NULLIF(statement_merchant,''), description_raw)), '[0-9]', ' '),
          '[^A-Z ]', ' '
        ),
        '\\s+', ' '
      )
    ) AS merchant_hint
  FROM FINANCE.STG.TRANSACTIONS t
),
mapped AS (
  SELECT
    b.*,
    m.merchant_canonical
  FROM base b
  LEFT JOIN FINANCE.CURATED.MERCHANT_CANONICAL_MAP m
    ON b.merchant_hint = m.merchant_hint
)
SELECT
  source, account_id, txn_date, post_date,
  description_raw, category_raw,
  amount_signed, is_payment,
  debit, credit,
  statement_merchant, country, chase_type,
  merchant_hint,
  COALESCE(merchant_canonical, 'UNKNOWN') AS merchant_final
FROM mapped;



In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE VIEW FINANCE.CURATED.SUBSCRIPTIONS_V4 AS
WITH c AS (
  SELECT * FROM FINANCE.CURATED.SUBSCRIPTIONS_V3
),
scored AS (
  SELECT
    *,
    DATEDIFF('day', last_seen, CURRENT_DATE()) AS days_since_last,
    IFF(std_gap_days IS NULL, 0, std_gap_days) AS std_gap_days_nz
  FROM c
)
SELECT *
FROM scored
WHERE days_since_last <= 90
  AND std_gap_days_nz <= 7
  AND median_amount >= 3
ORDER BY total_spend DESC;

SELECT merchant_hint, COUNT(*) ct, ROUND(SUM(amount_signed),2) spend
FROM FINANCE.CURATED.TRANSACTIONS_ENRICHED
WHERE merchant_final = 'UNKNOWN'
  AND amount_signed > 0
  AND NOT is_payment
GROUP BY 1
ORDER BY spend DESC
LIMIT 20;



In [ ]:
%%sql -r dataframe_4
